In [3]:
import pandas as pd
import tqdm
from google import genai
from dotenv import load_dotenv
from pinecone import Pinecone
import os

In [8]:
# retrieve necessary keys from .env
load_dotenv()
pinecone_key = os.getenv("Pinecone_key")
gemini_key = os.getenv("Gemini_key")

In [9]:
# hyperparameters
pinecone_top_k=2

In [10]:
# load clients
client = genai.Client(api_key=gemini_key)
pc = Pinecone(api_key=pinecone_key)

In [11]:
# retrieve top k results from pinecone output to create context
def retrieve_topk_text(results, top_k):
    '''
    return array of top_k texts
    '''
    return [hit["fields"]["chunk_text"].replace('\xad', '') for hit in (results['result']['hits'][:top_k])]

In [13]:
df = pd.read_csv(r"C:\Users\bcfra\Projects\Agents\Advising_Bot\data\stonybrook_pinecone_data.csv")

In [14]:
df['_id'] = df['_id'].astype(str)

In [15]:
df.head()

,_id,chunk_text
0,vec_1,Spring 2026 Undergraduate Catalog: The catalog...
1,vec_2,Catalog Search: Choose search categories to na...
2,vec_3,General Information: The Stony Brook Universit...
3,vec_4,Policies and Regulations: Students are respons...
4,vec_5,Colleges and Schools: The College of Arts and ...


In [16]:
records = df.to_dict("records")

In [17]:
index_name = "advising-bot"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

In [18]:
dense_index = pc.Index(index_name)

c:\Users\bcfra\anaconda3\envs\startenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
namespace = "bulletin"

batch_size = 50

for i in range(0, len(records), batch_size):
    batch = records[i: i + batch_size]
    dense_index.upsert_records(namespace, batch)

### Does flipped order and same order create the same embedding? 


In [29]:
query= "Tell me all the Asian American Studies courses?"

results = dense_index.search(
    namespace=namespace,
    query={
        "top_k": pinecone_top_k,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']:
        print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")

id: vec_1654 | score: 0.55  | text: AAS 311 - Asian American Studies: Theories and Methods: AAS 311 - Asian American Studies: Theories and MethodsThis course explores origins of Asian American studies, emerging trajectories, and core concepts, while introducing theories and methodologies for performing interdisciplinary, intersectional and comparative studies within the field. Analyzes topics such as race, gender, migration, diaspora, class, labor, sexuality, politics, and social justice as primary subjects of the discipline.3 creditsSBC: CER, USA
id: vec_1630 | score: 0.54  | text: AAS 212 - Asian and Asian American Studies Topics in the Humanities: AAS 212 - Asian and Asian American Studies Topics in the HumanitiesUsing methodologies of the Humanities disciplines, such as literature, linguistics, classics, cultural studies, philosophy, religious studies, art history and criticism, this course provides an introductory overview of important topics in Asian and Asian American Studies. T

In [30]:
pinecone_results = retrieve_topk_text(results, pinecone_top_k)

#### Build Gemini API as LLM backbone 

In [31]:
gemini_model = "gemini-3-flash-preview"
system_role= "You are a strict, factual AI assistant answering college students' answers. " \
            "Questions are answered using ONLY provided context. If the context does not contain the answer," \
            "you should answer: 'I do not have this information available'. " \
            "These context are provided to you right begore the user question under the <context> tags"

In [32]:
# all the context
formatted_context = " \n- ".join(pinecone_results)

In [33]:
prompt = f""" {system_role}
Below is the context:

<context>
- {formatted_context}
</context>
Question: {query}
"""

In [26]:
prompt

" You are a strict, factual AI assistant answering college students' answers. Questions are answered using ONLY provided context. If the context does not contain the answer,you should answer: 'I do not have this information available'. These context are provided to you right begore the user question under the <context> tags\nBelow is the context:\n\n<context>\n- SBU 101 - Introduction to Stony Brook: SBU 101 - Introduction to Stony BrookA seminar intended to integrate freshman students into the University community by providing information about Stony Brook and a forum for discussion of values, intellectual and social development, and personal as well as institutional expectations. This course is a graduation requirement for all first year students (students in their first year of college study). Not for credit in addition toADV 101, ACH 101, LDS 101, GLS 101, HDV 101, ITS 101, SSO 101,SCH 101, or LSE 101.1 credit,Prerequisite(s):First-semester freshmanGradedS/U grading \n- HON 101 - I

In [34]:
response = client.models.generate_content(model=gemini_model, contents=prompt)

In [35]:
print(response.text)

Based on the provided context, the Asian American Studies courses are:

- AAS 311 - Asian American Studies: Theories and Methods
- AAS 212 - Asian and Asian American Studies Topics in the Humanities
